# Lecture 2: Callbacks & Streaming (hands-on)
**Strands Agents with Ollama · Lecture 2 of 5** · companion to `Lecture02_Callbacks_and_Streaming.pdf`

You will:
1. Discover every event a callback handler receives
2. Write tool-tracking and buffered handlers
3. Run agents silently and capture results
4. Stream with `stream_async()` and serve a FastAPI endpoint

## 0. Setup: Ollama + Strands
Same steps as `StrandsAgent_SetUp.ipynb`. Run these once per Colab session (about 2 to 3 minutes).

> **Tip:** *Runtime → Change runtime type → T4 GPU* makes Ollama much faster. CPU works too, just slower.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start - check /tmp/ollama.log")

In [ ]:
!ollama pull qwen3:0.6b
# Optional, more reliable for tool use (1.4 GB):
# !ollama pull qwen3:1.7b

In [ ]:
!pip install -q "strands-agents[ollama,web-fetch]" fastapi uvicorn

### Shared model helper
Every lecture uses the same `make_model()` helper. Change `MODEL_ID` here to switch models for the whole notebook.

In [ ]:
from strands.models.ollama import OllamaModel

MODEL_ID = "qwen3:0.6b"          # try "qwen3:1.7b" for better tool use

def make_model(model_id=MODEL_ID, **kwargs):
    return OllamaModel(
        host="http://127.0.0.1:11434",
        model_id=model_id,
        options={"num_ctx": 8192},   # explicit context window (see Lecture 4)
        **kwargs,
    )

import importlib.metadata as md_
print("strands-agents", md_.version("strands-agents"), "| model:", MODEL_ID)

## 1. A calculator tool for this lab

In [ ]:
from strands import Agent, tool

@tool
def calculator(a: float, b: float, operation: str = "add") -> str:
    """Perform a math operation on two numbers.

    Args:
        a: First number
        b: Second number
        operation: One of "add", "subtract", "multiply", "divide", "power"
    """
    ops = {"add": a + b, "subtract": a - b, "multiply": a * b,
           "divide": a / b if b else "Error: division by zero", "power": a ** b}
    return str(ops.get(operation, f"Unknown operation: {operation}"))

## 2. Which events exist?
A callback handler is any function taking `**kwargs`. Let's record every key it ever receives.

In [ ]:
seen_keys = set()

def key_recorder(**kwargs):
    seen_keys.update(kwargs.keys())

agent = Agent(model=make_model(), tools=[calculator], callback_handler=key_recorder)
result = agent("Use the calculator to compute 2 to the power of 16.")
print(result)
print(sorted(seen_keys))

Key ones: `data` (text chunk), `current_tool_use` (tool call being built), `message` (complete message), `result` (final AgentResult).

## 3. A tool-tracking handler
`current_tool_use` fires **many times** while the tool input streams in, so de-duplicate by `toolUseId`.

In [ ]:
seen_tools = set()

def tracking_handler(**kwargs):
    if "data" in kwargs:
        print(kwargs["data"], end="", flush=True)
    tool_use = kwargs.get("current_tool_use") or {}
    if tool_use.get("name") and tool_use.get("toolUseId") not in seen_tools:
        seen_tools.add(tool_use["toolUseId"])
        print(f"\n[tool] calling {tool_use['name']} ...")

agent = Agent(model=make_model(), tools=[calculator], callback_handler=tracking_handler)
agent("Use the calculator to compute 2 to the power of 16.")

## 4. Buffered handler: complete messages only

In [ ]:
def buffered_handler(**kwargs):
    if "message" in kwargs and kwargs["message"].get("role") == "assistant":
        for block in kwargs["message"].get("content", []):
            if "text" in block:
                print("ASSISTANT:", block["text"].strip())
            if "toolUse" in block:
                print("TOOL CALL:", block["toolUse"]["name"], block["toolUse"]["input"])

agent = Agent(model=make_model(), tools=[calculator], callback_handler=buffered_handler)
agent("What is 2 to the power of 16, minus 1? Use the calculator.")

## 5. Silent mode
`callback_handler=None` prints nothing; the agent still runs and returns a result. Use it for sub-agents (Lecture 5), tests and batch jobs.

In [ ]:
agent = Agent(model=make_model(), tools=[calculator], callback_handler=None)
result = agent("What is 42 * 42? Use the calculator.")
print("Captured result:", result)
print("Stop reason:", result.stop_reason)

## 6. Async streaming with `stream_async()`
Colab and Jupyter support top-level `await`. In a `.py` script, wrap it with `asyncio.run(...)`.

In [ ]:
import time

async def stream_demo(prompt, **model_kwargs):
    agent = Agent(model=make_model(**model_kwargs), callback_handler=None)
    start, first_token = time.time(), None
    async for event in agent.stream_async(prompt):
        if "data" in event:
            first_token = first_token or time.time() - start
            print(event["data"], end="", flush=True)
        elif "result" in event:
            print(f"\n\n[done] stop_reason={event['result'].stop_reason}")
    print(f"time to first token: {first_token:.1f}s, total: {time.time()-start:.1f}s")

await stream_demo("Explain the agent loop in 3 sentences.")

In [ ]:
# Compare: thinking off -> first visible token arrives much sooner
await stream_demo("Explain the agent loop in 3 sentences.", additional_args={"think": False})

## 7. Serve a stream over HTTP (FastAPI)
`%%writefile` saves the cell as a Python file. We create **one agent per request** so users never share history.

In [ ]:
%%writefile fastapi_streaming.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from strands import Agent
from strands.models.ollama import OllamaModel

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/stream")
async def stream_response(request: PromptRequest):
    async def generate():
        model = OllamaModel(host="http://127.0.0.1:11434", model_id="qwen3:0.6b",
                            options={"num_ctx": 8192})
        agent = Agent(model=model, callback_handler=None)
        async for event in agent.stream_async(request.prompt):
            if "data" in event:
                yield event["data"]
    return StreamingResponse(generate(), media_type="text/plain")

In [ ]:
!nohup uvicorn fastapi_streaming:app --port 8000 > /tmp/api.log 2>&1 &
!sleep 5
!curl -N -X POST http://127.0.0.1:8000/stream -H "Content-Type: application/json" -d '{"prompt": "Give me 3 tips for writing good tool docstrings"}'

In [ ]:
!pkill -f "uvicorn fastapi_streaming" ; echo stopped

## 8. Your turn
**Exercise 1:** Build a *status line* handler: print `[thinking…]` once at the start, `[tool: name]` for each tool call, and then only the final answer text (hint: use the `message` event for the final text).

In [ ]:
def status_handler(**kwargs):
    # TODO
    ...

agent = Agent(model=make_model(), tools=[calculator], callback_handler=status_handler)
agent("What is 15% of 240? Use the calculator.")

**Exercise 2 (stretch):** Change the FastAPI endpoint to stream **newline-delimited JSON**: `{"type": "text", "data": ...}` for text and `{"type": "tool", "name": ...}` for tool calls. Test it with `curl -N`.

In [ ]:
# Your stretch solution here